In [3]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# Read in CSV files
brands = pd.read_csv('../_primary_system/data/brands.csv')
categories = pd.read_csv('../_primary_system/data/categories.csv')
customers = pd.read_csv('../_primary_system/data/customers.csv')
order_items = pd.read_csv('../_primary_system/data/order_items.csv')
orders = pd.read_csv('../_primary_system/data/orders.csv')
products = pd.read_csv('../_primary_system/data/products.csv')
staffs = pd.read_csv('../_primary_system/data/staffs.csv')
stocks = pd.read_csv('../_primary_system/data/stocks.csv')
stores = pd.read_csv('../_primary_system/data/stores.csv')

# Create database connection
connection = sqlite3.connect('store.db')

# Insert data into database
brands.to_sql('brands', connection, if_exists='replace', index=False)
categories.to_sql('categories', connection, if_exists='replace', index=False)
customers.to_sql('customers', connection, if_exists='replace', index=False)
order_items.to_sql('order_items', connection, if_exists='replace', index=False)
orders.to_sql('orders', connection, if_exists='replace', index=False)
products.to_sql('products', connection, if_exists='replace', index=False)
staffs.to_sql('staffs', connection, if_exists='replace', index=False)
stocks.to_sql('stocks', connection, if_exists='replace', index=False)
stores.to_sql('stores', connection, if_exists='replace', index=False)


3

# Data Diagram

  <img src="./diagram.png" width="500">

## SELECT & FROM
SELECT and FROM are the 2 minimum components we need to build a query in SQL. SELECT defines what columns we would like, and FROM defines the table we would like to pull data from.

In [5]:
# Run SQL query and load results into a DataFrame
query = """
SELECT 
    category_name 
FROM 
    categories;
"""

df = pd.read_sql_query(query, connection)
df

,category_name
0,Children Bicycles
1,Comfort Bicycles
2,Cruisers Bicycles
3,Cyclocross Bicycles
4,Electric Bikes
5,Mountain Bikes
6,Road Bikes


We can also use SELECT * to select all the columns in a table. Be careful when using this with large tables, as it will return all columns and rows for a table. It is advisable to specify the columns you would like and/or use this with LIMIT to avoid issues.

In [6]:
query = """
SELECT 
    * 
FROM 
    categories;
"""

df = pd.read_sql_query(query, connection)
df

,category_id,category_name
0,1,Children Bicycles
1,2,Comfort Bicycles
2,3,Cruisers Bicycles
3,4,Cyclocross Bicycles
4,5,Electric Bikes
5,6,Mountain Bikes
6,7,Road Bikes


## ORDER BY, DISTINCT, LIMIT
ORDER BY allows us to order the output of the query by 1 or more columns.

In [7]:
query = """
SELECT 
    first_name,
    last_name
FROM 
    customers
ORDER BY
    last_name;
"""

df = pd.read_sql_query(query, connection)
df

,first_name,last_name
0,Jamika,Acevedo
1,Penny,Acevedo
2,Ester,Acevedo
3,Shery,Acosta
4,Bettyann,Acosta
...,...,...
1440,Edda,Young
1441,Jasmin,Young
1442,Jayme,Zamora
1443,Alexandria,Zamora


In [8]:
query = """
SELECT 
    first_name,
    last_name
FROM 
    customers
ORDER BY
    last_name, first_name;
"""

df = pd.read_sql_query(query, connection)
df


,first_name,last_name
0,Ester,Acevedo
1,Jamika,Acevedo
2,Penny,Acevedo
3,Bettyann,Acosta
4,Shery,Acosta
...,...,...
1440,Edda,Young
1441,Jasmin,Young
1442,Alexandria,Zamora
1443,Jayme,Zamora


We can use ASC (Ascending) or DESC (Descending) to change the order used by ORDER BY.

DISTINCT will return only the unique values in a column, or the unique rows in the case of multiple columns. Below we use DISTINCT with last_name to get unique last names, but if we were to add first_name, it would give us all of the unique last_name - first_name pairs.  



In [9]:
query = """
SELECT 
    DISTINCT
    last_name
FROM 
    customers
ORDER BY 
    last_name DESC;
"""

df = pd.read_sql_query(query, connection)
df

,last_name
0,Zimmerman
1,Zamora
2,Young
3,Yates
4,Yang
...,...
748,Aguilar
749,Adkins
750,Adams
751,Acosta


LIMIT allows us to limit the number of output rows. This is useful when dealing with large tables where we only need to see the columns, for example.

In [10]:
query = """
SELECT 
    DISTINCT
    last_name
FROM 
    customers
ORDER BY 
    last_name ASC
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,last_name
0,Acevedo
1,Acosta
2,Adams
3,Adkins
4,Aguilar


LIMIT can be used in combination with OFFSET for pagination.
This skips the first 20 rows and returns the next 10.

In [11]:
query = """
SELECT 
    DISTINCT
    last_name
FROM 
    customers
ORDER BY 
    last_name ASC
LIMIT 10 OFFSET 20;
"""

df = pd.read_sql_query(query, connection)
df

,last_name
0,Baker
1,Baldwin
2,Ball
3,Ballard
4,Banks
5,Barber
6,Barnes
7,Barnett
8,Barr
9,Barrett


## WHERE, AND, OR
WHERE filters columns based on a conditional statement. This can be done with both numeric and text data.

In [12]:
query = """
SELECT 
    first_name,
    last_name
FROM 
    customers
WHERE
    last_name = 'Acevedo'
ORDER BY 
    first_name;
"""

df = pd.read_sql_query(query, connection)
df

,first_name,last_name
0,Ester,Acevedo
1,Jamika,Acevedo
2,Penny,Acevedo


We can add multiple conditions using AND and OR. When using multiple conditions, be sure that your logical statements are being properly interpreted. It is often useful to use parentheses to properly separate statements.

In [15]:
query = """
SELECT 
    *
FROM 
    customers
WHERE
    last_name = 'Acevedo'
AND
    first_name = 'Ester';
"""

df = pd.read_sql_query(query, connection)
df


,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1445,Ester,Acevedo,None,ester.acevedo@gmail.com,671 Miles Court,San Lorenzo,CA,94580


Note the difference in output between the 2 queries below.

In [16]:
query = """
SELECT 
    first_name,
    last_name
FROM 
    customers
WHERE
    last_name = 'Acevedo'
AND
    first_name = 'Ester'
OR
    first_name = 'Jamika';
"""

df = pd.read_sql_query(query, connection)
df

,first_name,last_name
0,Jamika,Acevedo
1,Jamika,Blanchard
2,Ester,Acevedo


In [17]:
query = """
SELECT 
    first_name,
    last_name
FROM 
    customers
WHERE
    last_name = 'Acevedo'
AND
    (
    (first_name = 'Ester')
    OR 
    (first_name = 'Jamika')
    );
"""

df = pd.read_sql_query(query, connection)
df


,first_name,last_name
0,Jamika,Acevedo
1,Ester,Acevedo


## IN, IS NULL, NOT
We can use IN within a conditional clause like WHERE or AND to filter a column based on whether or not it contains certain values. Here we will select all customers from New York and California only.

In [18]:
query = """
SELECT 
    *
FROM 
    customers
WHERE
    state IN ('CA', 'NY')
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df


,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,Debra,Burks,NaN,debra.burks@yahoo.com,9273 Thorne Ave.,Orchard Park,NY,14127
1,2,Kasha,Todd,NaN,kasha.todd@yahoo.com,910 Vine Street,Campbell,CA,95008
2,3,Tameka,Fisher,NaN,tameka.fisher@aol.com,769C Honey Creek St.,Redondo Beach,CA,90278
3,4,Daryl,Spence,NaN,daryl.spence@aol.com,988 Pearl Lane,Uniondale,NY,11553
4,5,Charolette,Rice,(916) 381-6003,charolette.rice@msn.com,107 River Dr.,Sacramento,CA,95820


We can use IS NULL to filter columns that have null values. Let's add this to the previous query to filter down to users with no phone number on file.

In [19]:
query = """
SELECT 
    *
FROM 
    customers
WHERE
    state IN ('CA', 'NY')
AND 
    phone IS NULL
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,Debra,Burks,None,debra.burks@yahoo.com,9273 Thorne Ave.,Orchard Park,NY,14127
1,2,Kasha,Todd,None,kasha.todd@yahoo.com,910 Vine Street,Campbell,CA,95008
2,3,Tameka,Fisher,None,tameka.fisher@aol.com,769C Honey Creek St.,Redondo Beach,CA,90278
3,4,Daryl,Spence,None,daryl.spence@aol.com,988 Pearl Lane,Uniondale,NY,11553
4,6,Lyndsey,Bean,None,lyndsey.bean@hotmail.com,769 West Road,Fairport,NY,14450


We can use NOT to reverse the logic of these, NOT IN and IS NOT NULL respectively, to get customers who are not in Texas or New York, and who do have a phone number.

In [20]:
query = """
SELECT 
    *
FROM 
    customers
WHERE
    state NOT IN ('CA', 'NY')
AND 
    phone IS NOT NULL
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,43,Mozelle,Carter,(281) 489-9656,mozelle.carter@aol.com,895 Chestnut Ave.,Houston,TX,77016
1,56,Lolita,Mosley,(281) 363-3309,lolita.mosley@hotmail.com,376 S. High Ridge St.,Houston,TX,77016
2,123,Robena,Hill,(361) 598-4414,robena.hill@hotmail.com,263 Cross St.,Corpus Christi,TX,78418
3,135,Dorthey,Jackson,(281) 926-8010,dorthey.jackson@msn.com,9768 Brookside St.,Houston,TX,77016
4,203,Minerva,Decker,(281) 271-6390,minerva.decker@yahoo.com,794 Greenrose Street,Houston,TX,77016


## LIKE
LIKE can be used to match patterns in text columns. This is done mainly with the % and _ operators.

% is used to match any sequence of 0 or more characters, while _ matches any single character. This can be used to find entries that start with, end with, or contain certain text.

In [24]:
query = """
SELECT 
    *
FROM 
    products
WHERE
    product_name LIKE 'T%'
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,product_id,product_name,brand_id,category_id,model_year,list_price
0,1,Trek 820 - 2016,9,6,2016,379.99
1,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99
2,7,Trek Slash 8 27.5 - 2016,9,6,2016,3999.99
3,8,Trek Remedy 29 Carbon Frameset - 2016,9,6,2016,1799.99
4,9,Trek Conduit+ - 2016,9,5,2016,2999.99


## Aggregate Functions
SQL functions are built-in operations that allow you to manipulate data, perform calculations, and retrieve information from the database. Aggregate functions perform calculations on sets of values and return a single value. The most commonly used aggregate functions include SUM(), AVG(), COUNT(), MIN(), and MAX().

Let's find the most expensive item purchased and the total items ordered using these.



In [25]:
query = """
SELECT 
    *,
    MAX(list_price)
FROM 
    order_items;
"""

df = pd.read_sql_query(query, connection)
df

,order_id,item_id,product_id,quantity,list_price,discount,MAX(list_price)
0,1364,2,155,2,11999.99,0.1,11999.99


The automatic name given to columns that result from functions are not always clear, so let's use AS to rename the column for our total order count query.

In [26]:
query = """
SELECT 
    SUM(quantity) AS total_items_ordered
FROM 
    order_items;
"""

df = pd.read_sql_query(query, connection)
df

,total_items_ordered
0,7078


Arithmetic Operators
Arithmetic operators are used to perform mathematical calculations on numeric values.

1. **Addition (+)**: Adds two numeric values together.
2. **Subtraction (-)**: Subtracts one numeric value from another.
3. **Multiplication (*)**: Multiplies two numeric values.
4. **Division (/)**: Divides one numeric value by another.
5. **Modulus (%)**: Returns the remainder after division.
6. **Unary Minus (-)**: Negates a numeric value (changes its sign).

Let's take the average of the list price multiplied by the discount in the order_items table to see the average discount in dollars.

In [27]:
query = """
SELECT 
    AVG(list_price * discount) AS avg_discount_usd
FROM 
    order_items;
"""

df = pd.read_sql_query(query, connection)
df

,avg_discount_usd
0,126.698149


Comparison Operators
Comparison operators are used to compare values and return a true or false result. We already learned one of these earlier, =, when using WHERE.

1. **Equal to (=)**: Checks if two values are equal.
2. **Not equal to (!= or <>)**: Checks if two values are not equal.
3. **Greater than (>)**: Checks if one value is greater than another.
4. **Less than (<)**: Checks if one value is less than another.
5. **Greater than or equal to (>=)**: Checks if one value is greater than or equal to another.
6. **Less than or equal to (<=)**: Checks if one value is less than or equal to another.

Let's expand on the last example and find the average final sale price for items priced at least at $1,000.

We can use the ROUND function here to round our output to 2 decimal places, since we are dealing with dollars.



In [28]:
query = """
SELECT 
    ROUND(AVG(list_price * (1 - discount)), 2) AS avg_saleprice_usd
FROM 
    order_items
WHERE
    list_price >= 1000;
"""

df = pd.read_sql_query(query, connection)
df

,avg_saleprice_usd
0,2542.02


## GROUP BY & HAVING
GROUP BY divides the rows returned from the SELECT statement into 1 or more groups. What makes this so powerful is that you can apply an aggregate function like SUM() to each group. This is often used to aggregate data by unique identifiers, time periods, categories, and more.

In [29]:
query = """
SELECT 
    order_id,
    ROUND(SUM(quantity * list_price * (1 - discount)), 2) final_sale_price
FROM 
    order_items
GROUP BY
    order_id;
"""

df = pd.read_sql_query(query, connection)
df

,order_id,final_sale_price
0,1,10231.05
1,2,1697.97
2,3,1519.98
3,4,1349.98
4,5,3900.06
...,...,...
1610,1611,8963.96
1611,1612,3781.13
1612,1613,5257.97
1613,1614,6104.04


HAVING functions like WHERE, except that it is used to filter on aggregated values such as those output by SUM() etc.

Let's use it in the same query above, to include only final sale prices greater than $20,000.

In ORDER BY and GROUP BY clauses, we can use positional references like 1 in place of the column name for better readability and ease of writing. The order is determined by the order of columns in the SELECT clause.

In [30]:
query = """
SELECT 
    order_id,
    ROUND(SUM(quantity * list_price * (1 - discount)), 2) final_sale_price
FROM 
    order_items
GROUP BY
    1
HAVING
    final_sale_price > 20000
ORDER BY
    2 DESC;
"""

df = pd.read_sql_query(query, connection)
df

,order_id,final_sale_price
0,1541,29147.03
1,937,27050.72
2,1506,25574.96
3,1482,25365.43
4,1364,24890.62
5,930,24607.03
6,1348,20648.95
7,1334,20509.43
8,973,20177.75


## CASE & CAST
We can use CASE to create new columns based on certain conditions. WHEN specifies the condition, THEN specifies the output when the condition is true, and ELSE specifies the output when the condition is false.

CASE is often used to create categories for existing columns, deal with NULL values, and within aggregate functions to perform conditional aggregation.

In [31]:
query = """
SELECT 
    *,
    CASE WHEN shipped_date > required_date THEN 1
         ELSE 0
         END AS 'shipped_late'
FROM 
    orders
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id,shipped_late
0,1,259,4,2016-01-01,2016-01-03,2016-01-03,1,2,0
1,2,1212,4,2016-01-01,2016-01-04,2016-01-03,2,6,0
2,3,523,4,2016-01-02,2016-01-05,2016-01-03,2,7,0
3,4,175,4,2016-01-03,2016-01-04,2016-01-05,1,3,1
4,5,1324,4,2016-01-03,2016-01-06,2016-01-06,2,6,0


CAST is used to change the data type of a column. This will be especially important later in this section when we perform joins, since the two columns being joined need to have the same data type.

In [32]:
query = """
SELECT 
    order_id,
    CAST(order_id AS float) AS order_id_float
FROM 
    orders
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df


,order_id,order_id_float
0,1,1.0
1,2,2.0
2,3,3.0
3,4,4.0
4,5,5.0


## INNER JOIN
In a relational database, data is typically distributed across multiple tables, as is the case with our sample database. To select complete data, you often need to query and combine data from multiple tables. We do this by using various table joins.

For joins to be possible, the tables in question must both share a column such as order_id. The query will then find the rows in both tables where the values are equal and join them.

An INNER JOIN keeps only the rows that have a match in both tables.

In [33]:
query = """
SELECT 
    *
FROM 
    orders
INNER JOIN
    customers
ON
    orders.customer_id = customers.customer_id
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,259,4,2016-01-01,2016-01-03,2016-01-03,1,2,259,Johnathan,Velazquez,None,johnathan.velazquez@hotmail.com,9680 E. Somerset Street,Pleasanton,CA,94566
1,2,1212,4,2016-01-01,2016-01-04,2016-01-03,2,6,1212,Jaqueline,Cummings,None,jaqueline.cummings@hotmail.com,478 Wrangler St.,Huntington Station,NY,11746
2,3,523,4,2016-01-02,2016-01-05,2016-01-03,2,7,523,Joshua,Robertson,None,joshua.robertson@gmail.com,81 Campfire Court,Patchogue,NY,11772
3,4,175,4,2016-01-03,2016-01-04,2016-01-05,1,3,175,Nova,Hess,None,nova.hess@msn.com,773 South Lafayette St.,Duarte,CA,91010
4,5,1324,4,2016-01-03,2016-01-06,2016-01-06,2,6,1324,Arla,Ellis,None,arla.ellis@yahoo.com,127 Crescent Ave.,Utica,NY,13501


If a customer_id was present in the orders table but not in the customer table, that row would be excluded. The opposite is true as well, if a customer_id was present in the customers table but not in the orders table, that customer's information would be excluded from the final output.

## LEFT JOIN
A LEFT JOIN works similarly to an INNER JOIN, except that it keeps all the records in the first table, even if there is no match. In the case of no match the data for the columns in the right table are filled with NULL values.

In [34]:
query = """
SELECT 
    *
FROM 
    orders
LEFT JOIN
    customers
ON
    orders.customer_id = customers.customer_id
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df


,order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,259,4,2016-01-01,2016-01-03,2016-01-03,1,2,259,Johnathan,Velazquez,None,johnathan.velazquez@hotmail.com,9680 E. Somerset Street,Pleasanton,CA,94566
1,2,1212,4,2016-01-01,2016-01-04,2016-01-03,2,6,1212,Jaqueline,Cummings,None,jaqueline.cummings@hotmail.com,478 Wrangler St.,Huntington Station,NY,11746
2,3,523,4,2016-01-02,2016-01-05,2016-01-03,2,7,523,Joshua,Robertson,None,joshua.robertson@gmail.com,81 Campfire Court,Patchogue,NY,11772
3,4,175,4,2016-01-03,2016-01-04,2016-01-05,1,3,175,Nova,Hess,None,nova.hess@msn.com,773 South Lafayette St.,Duarte,CA,91010
4,5,1324,4,2016-01-03,2016-01-06,2016-01-06,2,6,1324,Arla,Ellis,None,arla.ellis@yahoo.com,127 Crescent Ave.,Utica,NY,13501


Now, unlike in the INNER JOIN, if a customer_id was present in the orders table but not in the customers table, the rows in orders containing that id would be included in the final output.

## FULL OUTER JOIN
A FULL OUTER JOIN includes all the rows from both tables, even if there is no match. Like the LEFT JOIN, when there is no match for a row the data from the other table is filled with NULL values for that row.

In [35]:
query = """
SELECT 
    *
FROM 
    orders
FULL JOIN
    customers
ON
    orders.customer_id = customers.customer_id
LIMIT 5;
"""

df = pd.read_sql_query(query, connection)
df

,order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,259,4,2016-01-01,2016-01-03,2016-01-03,1,2,259,Johnathan,Velazquez,None,johnathan.velazquez@hotmail.com,9680 E. Somerset Street,Pleasanton,CA,94566
1,2,1212,4,2016-01-01,2016-01-04,2016-01-03,2,6,1212,Jaqueline,Cummings,None,jaqueline.cummings@hotmail.com,478 Wrangler St.,Huntington Station,NY,11746
2,3,523,4,2016-01-02,2016-01-05,2016-01-03,2,7,523,Joshua,Robertson,None,joshua.robertson@gmail.com,81 Campfire Court,Patchogue,NY,11772
3,4,175,4,2016-01-03,2016-01-04,2016-01-05,1,3,175,Nova,Hess,None,nova.hess@msn.com,773 South Lafayette St.,Duarte,CA,91010
4,5,1324,4,2016-01-03,2016-01-06,2016-01-06,2,6,1324,Arla,Ellis,None,arla.ellis@yahoo.com,127 Crescent Ave.,Utica,NY,13501


Again, here all the rows from both orders and customers would be included, even where there was no match.

## SELF JOIN
A SELF JOIN is a regular join that joins a table to itself. Typically, these are used to query hierarchical data or to compare rows within the same table.

To create a SELF JOIN, you use the same table twice with different table aliases.

One example of where you can apply a SELF JOIN is pulling the most recent and second most recent purchase dates for customers with at least 2 orders. A more complex version of this could be used to calculate the average days between transactions, which is often used to segment customers in e-commerce.

In [36]:
query = """
SELECT 
    t1.customer_id,
    MAX(t1.order_date) most_recent_order,
    MAX(t2.order_date) second_most_recent_order
FROM 
    orders t1
INNER JOIN
    orders t2
ON 
    t1.customer_id = t2.customer_id
AND
    t1.order_date > t2.order_date
GROUP BY
    1;
"""

df = pd.read_sql_query(query, connection)
df

,customer_id,most_recent_order,second_most_recent_order
0,1,2018-11-18,2018-04-18
1,2,2018-04-09,2017-08-21
2,3,2018-10-21,2018-04-06
3,4,2018-04-18,2017-11-21
4,5,2018-04-17,2016-11-24
...,...,...,...
126,231,2018-04-26,2017-12-14
127,233,2018-04-13,2017-06-03
128,237,2018-04-08,2016-04-11
129,239,2018-04-29,2017-08-09


## UNION
With joins, we have been combining data horizontally, but with UNION we combine it vertically. UNION combines the results of 2 or more SELECT statements.

The conditions for this are that the number, order, and data types of the columns are the same. UNION removes duplicate rows. In the case you wish to keep these UNION ALL should be used instead.

As an example, let's pretend that our store has an extremely large product catalog, and keeps their products in different tables separated by year. We want to get the products for 2016 and 2019, so we can use UNION to combine the results.



In [37]:
query = """
SELECT 
    product_id,
    product_name,
    model_year,
    list_price
FROM 
    products
WHERE
    model_year = 2016
UNION
SELECT 
    product_id,
    product_name,
    model_year,
    list_price
FROM 
    products
WHERE
    model_year = 2019;
"""

df = pd.read_sql_query(query, connection)
df

,product_id,product_name,model_year,list_price
0,1,Trek 820 - 2016,2016,379.99
1,2,Ritchey Timberwolf Frameset - 2016,2016,749.99
2,3,Surly Wednesday Frameset - 2016,2016,999.99
3,4,Trek Fuel EX 8 29 - 2016,2016,2899.99
4,5,Heller Shagamaw Frame - 2016,2016,1320.99
5,6,Surly Ice Cream Truck Frameset - 2016,2016,469.99
6,7,Trek Slash 8 27.5 - 2016,2016,3999.99
7,8,Trek Remedy 29 Carbon Frameset - 2016,2016,1799.99
8,9,Trek Conduit+ - 2016,2016,2999.99
9,10,Surly Straggler - 2016,2016,1549.00


## Subquery
Subqueries are queries nested within our main query. This is useful when we want to use the output of 1 query to filter another or calculate values based on a subset of data.

One example is if we want to see 2019 products with an above average price.

In [38]:
query = """
SELECT 
    *
FROM 
    products
WHERE
    model_year = 2019
AND
    list_price > (
                  SELECT
                      AVG(list_price)
                  FROM
                      products
                  WHERE
                      model_year = 2019
                 );
"""

df = pd.read_sql_query(query, connection)
df

,product_id,product_name,brand_id,category_id,model_year,list_price
0,319,Trek Checkpoint SL 5 Women's - 2019,9,7,2019,2799.99
1,320,Trek Checkpoint SL 6 - 2019,9,7,2019,3799.99
2,321,Trek Checkpoint ALR Frameset - 2019,9,7,2019,3199.99


IN can also be used to filter the outer query when using the inner(sub-query) query. Here we will use a subquery that returns order ids for the Trek brand that were discounted at least 20%, and use that to filter orders and return the unique order_id and customer_id for orders with those conditions.

Note here that DISTINCT returns unique pairs of order_id and customer_id, so there will be duplicate customer ids for customers with more than 1 order.

In [39]:
query = """
SELECT 
    DISTINCT
    order_id,
    customer_id
FROM 
    orders
WHERE
    order_id IN (
                  SELECT
                      DISTINCT
                      order_id
                  FROM
                      order_items
                  INNER JOIN
                      products
                  ON
                      order_items.product_id = products.product_id
                  AND
                      brand_id = 9
                  AND
                      discount >= .20
                 );
"""

df = pd.read_sql_query(query, connection)
df


,order_id,customer_id
0,1,259
1,16,552
2,19,696
3,28,252
4,29,437
...,...,...
293,1601,43
294,1602,55
295,1605,123
296,1610,15


## EXISTS
EXISTS checks if there are any rows present in a subquery, and returns true if the subquery either returns rows or returns NULL. This can be used with NOT to achieve the opposite effect. To achieve this a column in the subquery needs to be matched with a column in the main query; this is known as a correlated subquery.

Since the result is true if any rows are returned, 1 is often used in the SELECT statement since the columns are irrelevant.

In [40]:
query = """
SELECT 
    DISTINCT
    order_id,
    customer_id
FROM 
    orders
WHERE
    EXISTS (
            SELECT
                1
            FROM
                order_items
            WHERE
                discount >= .20
            AND
                order_items.order_id = orders.order_id
            );
"""

df = pd.read_sql_query(query, connection)
df


,order_id,customer_id
0,1,259
1,11,1326
2,15,450
3,16,552
4,17,1175
...,...,...
855,1610,15
856,1611,6
857,1612,3
858,1614,135


https://www.kaggle.com/code/dillonmyrick/sql-beginner-to-advanced-with-practical-examples